In [1]:
import sys 

sys.path.append("../..")

In [11]:
import numpy as np 
from tqdm import tqdm 
import torch
from torch import nn
from torch.nn import functional as F
from xaikd import models, datasets, bases

from pathlib import Path

from xaikd.utils import metrics


In [3]:
model = models.get_model("cifar100-resnet18-p1")

In [4]:
class SubResnet(nn.Module):
    def __init__(self, model):
        super().__init__()
        
        self.model = model 
        
    def forward(self, x):
        # do something 
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        
        logits = self.model.layer4(x)
        logits = self.model.avgpool(logits)
        logits = torch.flatten(logits, 1)
        logits = self.model.fc(logits)
        
        return x, logits

In [5]:
@torch.no_grad()
def ano():
    def fh(module, inp, output):
        setattr(module, "__output", output)
    
    
    subresnet = SubResnet(model)
    
    try :
        
        module = model.layer3[-1] 
        hook = module.register_forward_hook(fh)
        
        x = torch.randn((10, 3, 32, 32))
        
        expected_logits = model(x)
        
        expected_act = getattr(module, "__output")
        
        actual_act, actual_logits = subresnet(x)
        
        np.testing.assert_allclose(actual_act, expected_act)
        np.testing.assert_allclose(actual_logits, expected_logits)

    finally:
        hook.remove()
        
    print("sanity check passed!")
ano()

sanity check passed!


In [6]:
subresnet = SubResnet(model)

In [19]:
np.random.seed(1)

num_training_samples = 500

ds = datasets.construct("cifar100-people", num_training_samples=num_training_samples)

train_loader = ds.loader(train_split=True, shuffle=True)
val_loader = ds.loader(train_split=False, shuffle=False)

We are building `cifar100-people` containing 5 fine classes
> baby (2)
> boy (11)
> girl (35)
> man (46)
> woman (98)


# Setup Param

In [83]:
basis_name = "pca--centered"
# basis_name = "prca-abs--centered"

K = 100


basis = bases.get_basis(basis_name)
basis.load(
    artifact_dir=Path(
        f"../../artifacts/2023-06-s8/local/accuracy-basis/cifar100-people--n{num_training_samples}/logit-mod-oneclass/cifar100-resnet18-p1/layer3"
    )
)


projector = basis.construct_projection_on_rank_k(k=K, device="cpu")

# Training SVR

In [84]:
@torch.no_grad()
def collect_feat_and_output(loader):
    
    arr_x = []
    
    arr_y = []
    
    
    for i, (x, _) in tqdm(enumerate(loader)):
        
        act, logits = subresnet(x)
        
        projected_act = projector(act)
        shape = projected_act.shape[1:]
        spatial_shape = shape[1:]
        
        arr_x.append(
            F.avg_pool2d(projected_act, spatial_shape).flatten(start_dim=1).numpy()
        )
        arr_y.append(
            logits.numpy()[:, ds.selected_classes]
        )
        
    arr_x = np.vstack(arr_x)
    arr_y = np.vstack(arr_y)
    
    return arr_x, arr_y, shape
    
X, y, shape = collect_feat_and_output(train_loader)    

40it [00:26,  1.50it/s]


In [85]:
X.shape, shape

((2500, 100), torch.Size([100, 8, 8]))

In [86]:
from sklearn import svm
from sklearn.multioutput import MultiOutputRegressor

def fit_svm(X, y):
    regr = svm.SVR()

    regr = MultiOutputRegressor(regr)
    regr.fit(X, y)
    
    return regr

approxer = fit_svm(X, y)

In [87]:
from torch.nn import functional as F 

class SVCConvLayer(nn.Module):
    
    def __init__(self, approxer, X_training):
        super().__init__()
        
        ntraining, d = X_training.shape
        
        k = len(approxer.estimators_)
        A = np.zeros((k, ntraining))

        intercepts = np.zeros((k, ))
        
        for i in range(k):
            _est = approxer.estimators_[i]
            A[i, _est.support_] = _est.dual_coef_ 
            intercepts[i] = _est._intercept_
        
        self.A = torch.from_numpy(A).float()
        self.intercepts = torch.from_numpy(intercepts).float().reshape((1, k))
        
  
        self.gamma = 1 / (d * X_training.var())
        
        support_vectors = torch.from_numpy(X_training).float()
        
        self.W = support_vectors
        self.b = torch.linalg.norm(support_vectors, dim=1, keepdim=True) ** 2
        self.b = self.b.T
        
#         print("A.shape", self.A.shape)
#         print("W.shape", self.W.shape)     
#         print("b.shape", self.b.shape)     

    
    def forward(self, x):
        
        assert len(x.shape) == 2
        
        n, d = x.shape
        
        norm =  torch.norm(x, dim=1, keepdim=True)

#         print("x.shape", x.shape)
#         print("norm.shape", norm.shape)
#         print("b.shape", self.b.shape)

        out = x @ self.W.T
    
#         print("out.shape", out.shape)
        
        out = (2*out - norm ** 2 - self.b) * self.gamma
        out = torch.exp(out)
        
        out = out @ self.A.T
        
#         print("out.shape", out.shape)

        out = out + self.intercepts


        return out
    
svc_module = SVCConvLayer(approxer, X)

@torch.no_grad()
def sanity_check():

    X, y = next(iter(val_loader))
    
    act, _ = subresnet(X)
    
    projected_act = projector(act)
    
    pooled_act = F.avg_pool2d(projected_act, shape[1:]).flatten(start_dim=1)
        
    actual = svc_module(pooled_act).numpy()
    
    expected = approxer.predict(pooled_act)
    
    print(actual.shape, expected.shape)
    
    np.testing.assert_allclose(
        actual,
        expected,
        atol=1e-3
    )
    
    print("sanity check passed!")
    
    
sanity_check()

(64, 5) (64, 5)
sanity check passed!


In [88]:
class ApproxResnet(nn.Module):
    def __init__(self, model):
        super().__init__()
        
        self.model = model 
        
    def forward(self, x):
        # do something 
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        
        x = projector(x)
                
        x = svc_module(F.avg_pool2d(x, shape[1:]).flatten(start_dim=1))

        # this makes thing compatible with metrics.accuracy_with_subclasses
        logits = torch.zeros(x.shape[0], 100)
        
        logits[:, ds.selected_classes] = x

        return logits
    
approx_resnet = ApproxResnet(model)

In [89]:
acc = metrics.accuracy_with_subclasses(approx_resnet, val_loader, ds.selected_classes, ds.transform_target, device="cpu")

print(f"- {basis_name}(N={num_training_samples}, K={K}): acc={acc:.4f}")

- pca--centered(N=500, K=100): acc=0.4940


In [90]:
metrics.accuracy_with_subclasses(model, val_loader, ds.selected_classes, ds.transform_target, device="cpu")

0.550000011920929

basisline=0.55

- prca-abs--centered(N=50, K=10): acc=0.3760
- pca--centered(N=50, K=10): acc=0.3080

- pca--centered(N=50, K=25): acc=0.3680
- prca-abs--centered(N=50, K=25): acc=0.3780

- pca--centered(N=50, K=100): acc=0.4040
- prca-abs--centered(N=50, K=100): acc=0.4100

----

- pca--centered(N=500, K=10): acc=0.3760
- prca-abs--centered(N=500, K=10): acc=0.4060
- pca--centered(N=500, K=25): acc=0.4520
- prca-abs--centered(N=500, K=25): acc=0.4960
- pca--centered(N=500, K=100): acc=0.4940
- prca-abs--centered(N=500, K=100): acc=0.5000




